In [9]:
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import numpy as np


In [11]:

def load_period_data(period_id, folder="../data/processed"):
    train_path = f"{folder}/snn_train_period_{period_id}.csv"
    test_path  = f"{folder}/snn_test_period_{period_id}.csv"
    
    df_train = pd.read_csv(train_path)
    df_test  = pd.read_csv(test_path)
    
    return df_train, df_test

def filter_by_stock(df, stock_ticker):
    return df[df['ticker'] == stock_ticker].reset_index(drop=True)

def prepare_X_y(df, target_col="target"):
    X = df.drop(columns=[target_col, 'ticker', 'date'], errors='ignore')
    y = df[target_col] + 1  # Convertimos -1,0,1 → 0,1,2
    y_cat = to_categorical(y, num_classes=3)
    return X, y_cat, y  # devolvemos también y sin one-hot para métricas

def build_shallow_nn(input_dim, learning_rate=0.001):
    model = Sequential([
        Dense(31, activation='relu', input_dim=input_dim),
        Dense(16, activation='relu'),
        Dense(3, activation='softmax')
    ])
    model.compile(optimizer=Adam(learning_rate=learning_rate),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model


def load_and_filter(period_id, stock_ticker=None, folder="../data/processed"):
    df_train, df_test = load_period_data(period_id, folder)
    #if stock_ticker is not None:
    #    df_train = filter_by_stock(df_train, stock_ticker)
    #    df_test  = filter_by_stock(df_test, stock_ticker)
    return df_train, df_test


def prepare_data(df_train, df_test, target_col="target"):
    X_train, y_train_cat, y_train, scaler = prepare_X_y(df_train, target_col=target_col)
    X_test, y_test_cat, y_test, _ = prepare_X_y(df_test, target_col=target_col, scaler=scaler)
    return X_train, y_train_cat, y_train, X_test, y_test_cat, y_test, scaler


def train_model(X_train, y_train_cat, X_val, y_val_cat, input_dim=None, epochs=20, batch_size=32, use_focal=True):
    if input_dim is None:
        input_dim = X_train.shape[1]
    model = build_shallow_nn(input_dim=input_dim, use_focal=use_focal)
    model.fit(X_train, y_train_cat, validation_data=(X_val, y_val_cat),
              epochs=epochs, batch_size=batch_size, verbose=2)
    return model

def predict_model(model, X_test):
    y_pred_probs = model.predict(X_test)
    y_pred = np.argmax(y_pred_probs, axis=1)
    return y_pred, y_pred_probs


def evaluate_metrics(y_test, y_pred, y_pred_probs):
    print("\n=== Confusion Matrix ===")
    print(confusion_matrix(y_test, y_pred))
    
    print("\n=== Classification Report ===")
    print(classification_report(y_test, y_pred, target_names=['SELL','HOLD','BUY']))
    
    try:
        auc = roc_auc_score(to_categorical(y_test, num_classes=3), y_pred_probs, multi_class='ovr')
        print(f"\nAUC (one-vs-rest): {auc:.4f}")
    except Exception as e:
        print("Error calculando AUC:", e)


def run_shallow_nn(period_id, stock_ticker=None, epochs=20, batch_size=32, use_focal=True):
  
    df_train, df_test = load_and_filter(period_id, stock_ticker)
    

    X_train, y_train_cat, y_train, X_test, y_test_cat, y_test, scaler = prepare_data(df_train, df_test)
    

    model = train_model(X_train, y_train_cat, X_test, y_test_cat, epochs=epochs, batch_size=batch_size, use_focal=use_focal)
    

    y_pred, y_pred_probs = predict_model(model, X_test)
    
    evaluate_metrics(y_test, y_pred, y_pred_probs)
    
    return model, scaler

In [12]:
period_id = 0
stock_ticker = "AAPL"
model = train_and_evaluate_stock_nn(period_id, stock_ticker, epochs=50)

Epoch 1/50


/Users/sergio/Documents/NeuralNetworks/NeuralNetworks-Group1/venv/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9724/9724 ━━━━━━━━━━━━━━━━━━━━ 3s 296us/step - accuracy: 0.3994 - loss: 1.0810 - val_accuracy: 0.4803 - val_loss: 1.0612
Epoch 2/50
9724/9724 ━━━━━━━━━━━━━━━━━━━━ 3s 296us/step - accuracy: 0.4143 - loss: 1.0714 - val_accuracy: 0.4340 - val_loss: 1.0894
Epoch 3/50
9724/9724 ━━━━━━━━━━━━━━━━━━━━ 3s 297us/step - accuracy: 0.4213 - loss: 1.0677 - val_accuracy: 0.4768 - val_loss: 1.0497
Epoch 4/50
9724/9724 ━━━━━━━━━━━━━━━━━━━━ 3s 291us/step - accuracy: 0.4229 - loss: 1.0663 - val_accuracy: 0.4828 - val_loss: 1.0456
Epoch 5/50
9724/9724 ━━━━━━━━━━━━━━━━━━━━ 3s 295us/step - accuracy: 0.4241 - loss: 1.0652 - val_accuracy: 0.4643 - val_loss: 1.0597
Epoch 6/50
9724/9724 ━━━━━━━━━━━━━━━━━━━━ 3s 294us/step - accuracy: 0.4247 - loss: 1.0645 - val_accuracy: 0.4682 - val_loss: 1.0612
Epoch 7/50
9724/9724 ━━━━━━━━━━━━━━━━━━━━ 3s 295us/step - accuracy: 0.4261 - loss: 1.0639 - val_accuracy: 0.4796 - val_loss: 1.0513
Epoch 8/50
9724/9724 ━━━━━━━━━━━━━━━━━━━━ 3s 295us/step - accuracy: 0.4271 - loss: 1.06